In [2]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from scipy.sparse import load_npz
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import TruncatedSVD

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

train_tfidf = load_npz("train_tfidf.npz")
valid_tfidf = load_npz("valid_tfidf.npz")
test_tfidf = load_npz("test_tfidf.npz")

train_df = pd.read_csv("train_features.csv")
valid_df = pd.read_csv("valid_features.csv")
test_df = pd.read_csv("test_features.csv")

y_train = train_df["label"].astype(int).values
y_valid = valid_df["label"].astype(int).values
y_test = test_df["label"].astype(int).values

numeric_cols = train_df.columns.difference(["text", "tokens", "pos_seq", "label"])
X_train_extra = train_df[numeric_cols].astype(np.float64).values
X_valid_extra = valid_df[numeric_cols].astype(np.float64).values
X_test_extra = test_df[numeric_cols].astype(np.float64).values

print("Applying dimensionality reduction...")
n_components = 300
svd = TruncatedSVD(n_components=n_components, random_state=42)
X_train_tfidf = svd.fit_transform(train_tfidf)
X_valid_tfidf = svd.transform(valid_tfidf)
X_test_tfidf = svd.transform(test_tfidf)

print(f"Reduced TF-IDF shape: {X_train_tfidf.shape}")
print(f"Extra features shape: {X_train_extra.shape}")
print(f"Explained variance ratio: {svd.explained_variance_ratio_.sum():.4f}")

scaler_tfidf = StandardScaler()
X_train_tfidf = scaler_tfidf.fit_transform(X_train_tfidf)
X_valid_tfidf = scaler_tfidf.transform(X_valid_tfidf)
X_test_tfidf = scaler_tfidf.transform(X_test_tfidf)

scaler_extra = StandardScaler()
X_train_extra = scaler_extra.fit_transform(X_train_extra)
X_valid_extra = scaler_extra.transform(X_valid_extra)
X_test_extra = scaler_extra.transform(X_test_extra)

class TextDataset(Dataset):
    def __init__(self, tfidf_features, extra_features, labels):
        self.tfidf = torch.FloatTensor(tfidf_features)
        self.extra = torch.FloatTensor(extra_features)
        self.labels = torch.LongTensor(labels)
    
    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, idx):
        return self.tfidf[idx], self.extra[idx], self.labels[idx]
batch_size = 128
train_dataset = TextDataset(X_train_tfidf, X_train_extra, y_train)
valid_dataset = TextDataset(X_valid_tfidf, X_valid_extra, y_valid)
test_dataset = TextDataset(X_test_tfidf, X_test_extra, y_test)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

class TextCNN(nn.Module):
    def __init__(self, input_size, extra_features_size, num_filters=100, filter_sizes=[3, 4, 5], dropout=0.5):
        super(TextCNN, self).__init__()
        
        self.input_size = input_size
        self.convs = nn.ModuleList([
            nn.Conv1d(in_channels=1, out_channels=num_filters, kernel_size=fs)
            for fs in filter_sizes
        ])
        
        self.dropout = nn.Dropout(dropout)
        total_filters = num_filters * len(filter_sizes)
        self.fc1 = nn.Linear(total_filters + extra_features_size, 128)
        self.fc2 = nn.Linear(128, 2)
        
        self.relu = nn.ReLU()
    
    def forward(self, tfidf, extra):
        x = tfidf.unsqueeze(1)
        
        conv_outputs = []
        for conv in self.convs:
            conv_out = self.relu(conv(x))
         
            pooled = torch.max(conv_out, dim=2)[0]
            conv_outputs.append(pooled)
        
        conv_concat = torch.cat(conv_outputs, dim=1)
        conv_concat = self.dropout(conv_concat)
        
        combined = torch.cat([conv_concat, extra], dim=1)
        
        x = self.relu(self.fc1(combined))
        x = self.dropout(x)
        x = self.fc2(x)
        
        return x

input_size = X_train_tfidf.shape[1]
extra_size = X_train_extra.shape[1]

model = TextCNN(
    input_size=input_size,
    extra_features_size=extra_size,
    num_filters=100,
    filter_sizes=[3, 4, 5],
    dropout=0.5
).to(device)

print(f"\nModel architecture:")
print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=0.0001)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=3)

def train_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    for tfidf, extra, labels in loader:
        tfidf, extra, labels = tfidf.to(device), extra.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(tfidf, extra)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    
    return total_loss / len(loader), correct / total

def evaluate(model, loader):
    model.eval()
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for tfidf, extra, labels in loader:
            tfidf, extra, labels = tfidf.to(device), extra.to(device), labels.to(device)
            outputs = model(tfidf, extra)
            _, predicted = torch.max(outputs, 1)
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    return np.array(all_preds), np.array(all_labels)

num_epochs = 30
best_val_acc = 0
patience = 5
patience_counter = 0

print("\n" + "="*50)
print("TRAINING CNN MODEL")
print("="*50)

for epoch in range(num_epochs):
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer)
    val_preds, val_labels = evaluate(model, valid_loader)
    val_acc = accuracy_score(val_labels, val_preds)
    
    scheduler.step(val_acc)
    
    print(f"Epoch {epoch+1}/{num_epochs}")
    print(f"  Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}")
    print(f"  Valid Acc: {val_acc:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), 'best_cnn_model.pth')
        patience_counter = 0
        print(f"  → New best model saved!")
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"\nEarly stopping triggered after {epoch+1} epochs")
            break

model.load_state_dict(torch.load('best_cnn_model.pth'))

print("\n" + "="*50)
print("FINAL EVALUATION ON TEST SET")
print("="*50)

test_preds, test_labels = evaluate(model, test_loader)
test_acc = accuracy_score(test_labels, test_preds)

print(f"Best Validation Accuracy: {best_val_acc:.4f}")
print(f"\nFINAL TEST ACCURACY: {test_acc:.4f}")
print("\nClassification Report:")
print(classification_report(test_labels, test_preds))

print("\n" + "="*50)
print("MODEL SUMMARY")
print("="*50)
print(f"Input dimension: {input_size}")
print(f"Extra features: {extra_size}")
print(f"Filter sizes: [3, 4, 5]")
print(f"Number of filters per size: 100")
print(f"Total convolutional filters: 300")
print(f"Dropout rate: 0.5")
print(f"Batch size: {batch_size}")
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")

Using device: cpu
Applying dimensionality reduction...
Reduced TF-IDF shape: (21464, 300)
Extra features shape: (21464, 59)
Explained variance ratio: 0.2644

Model architecture:
TextCNN(
  (convs): ModuleList(
    (0): Conv1d(1, 100, kernel_size=(3,), stride=(1,))
    (1): Conv1d(1, 100, kernel_size=(4,), stride=(1,))
    (2): Conv1d(1, 100, kernel_size=(5,), stride=(1,))
  )
  (dropout): Dropout(p=0.5, inplace=False)
  (fc1): Linear(in_features=359, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=2, bias=True)
  (relu): ReLU()
)

Total parameters: 47,838

TRAINING CNN MODEL
Epoch 1/30
  Train Loss: 0.6765, Train Acc: 0.5856
  Valid Acc: 0.6844
  → New best model saved!
Epoch 2/30
  Train Loss: 0.6412, Train Acc: 0.6318
  Valid Acc: 0.6718
Epoch 3/30
  Train Loss: 0.6272, Train Acc: 0.6494
  Valid Acc: 0.6872
  → New best model saved!
Epoch 4/30
  Train Loss: 0.6197, Train Acc: 0.6575
  Valid Acc: 0.6913
  → New best model saved!
Epoch 5/30
  Train Loss: 0.61